In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [29]:
df = pd.read_csv("mnist_784.csv")

X = df.drop("class", axis=1).values / 255.0
y = df["class"].astype(int).values

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [31]:
def one_hot(y):
    oh = np.zeros((y.shape[0], 10))
    oh[np.arange(y.shape[0]), y] = 1
    return oh

y_train_oh = one_hot(y_train)
y_test_oh = one_hot(y_test)

In [32]:
def init_params(layers):
    W = []
    b = []
    for i in range(len(layers)-1):
        W.append(np.random.randn(layers[i], layers[i+1]) * 0.01)
        b.append(np.zeros((1, layers[i+1])))
    return W, b

In [33]:
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

In [34]:
def forward(X, W, b):
    A = [X]
    Z = []
    
    for i in range(len(W)-1):
        z = A[-1] @ W[i] + b[i]
        a = relu(z)
        Z.append(z)
        A.append(a)
    
    z = A[-1] @ W[-1] + b[-1]
    a = softmax(z)
    
    Z.append(z)
    A.append(a)
    
    return A, Z

In [35]:
def loss(y, y_hat):
    m = y.shape[0]
    return -np.sum(y * np.log(y_hat + 1e-8)) / m

In [36]:
def backward(A, Z, W, y):
    dW = []
    db = []
    
    m = y.shape[0]
    dz = A[-1] - y
    
    for i in reversed(range(len(W))):
        dw = A[i].T @ dz / m
        d_b = np.sum(dz, axis=0, keepdims=True) / m
        
        dW.insert(0, dw)
        db.insert(0, d_b)
        
        if i > 0:
            dz = (dz @ W[i].T) * relu_deriv(Z[i-1])
    
    return dW, db

In [37]:
def update(W, b, dW, db, lr):
    for i in range(len(W)):
        W[i] -= lr * dW[i]
        b[i] -= lr * db[i]
    return W, b

In [38]:
def train(X, y, layers, epochs=10, lr=0.01):
    W, b = init_params(layers)
    
    for i in range(epochs):
        A, Z = forward(X, W, b)
        l = loss(y, A[-1])
        
        dW, db = backward(A, Z, W, y)
        W, b = update(W, b, dW, db, lr)
        
        print("Epoch", i+1, "Loss:", l)
    
    return W, b

In [39]:
def accuracy(X, y, W, b):
    A, _ = forward(X, W, b)
    pred = np.argmax(A[-1], axis=1)
    return np.mean(pred == y)

In [40]:
layers = [784, 128, 64, 10]

W, b = train(X_train, y_train_oh, layers, epochs=10, lr=0.01)

acc = accuracy(X_test, y_test, W, b)
print("Accuracy:", acc)

Epoch 1 Loss: 2.3025905473715476
Epoch 2 Loss: 2.302587168244971
Epoch 3 Loss: 2.3025837947056393
Epoch 4 Loss: 2.302580426394545
Epoch 5 Loss: 2.302577063478566
Epoch 6 Loss: 2.3025737061796723
Epoch 7 Loss: 2.3025703542409977
Epoch 8 Loss: 2.3025670078003255
Epoch 9 Loss: 2.3025636669539704
Epoch 10 Loss: 2.302560331805997
Accuracy: 0.11242857142857143


In [41]:
configs = [
    [784, 64, 10],
    [784, 128, 10],
    [784, 128, 64, 10]
]

for c in configs:
    print("\nLayers:", c)
    W, b = train(X_train, y_train_oh, c, epochs=5, lr=0.01)
    print("Accuracy:", accuracy(X_test, y_test, W, b))


Layers: [784, 64, 10]
Epoch 1 Loss: 2.3021713309212566
Epoch 2 Loss: 2.3021185169868144
Epoch 3 Loss: 2.3020656857160766
Epoch 4 Loss: 2.302012823752314
Epoch 5 Loss: 2.3019599327374505
Accuracy: 0.14785714285714285

Layers: [784, 128, 10]
Epoch 1 Loss: 2.3016470779650495
Epoch 2 Loss: 2.301527417648597
Epoch 3 Loss: 2.3014076834629953
Epoch 4 Loss: 2.3012878677401445
Epoch 5 Loss: 2.3011679502101723
Accuracy: 0.11764285714285715

Layers: [784, 128, 64, 10]
Epoch 1 Loss: 2.3025717964733583
Epoch 2 Loss: 2.3025684276507206
Epoch 3 Loss: 2.302565064192833
Epoch 4 Loss: 2.302561706283378
Epoch 5 Loss: 2.302558353844367
Accuracy: 0.12792857142857142
